In [ ]:
import pandas as pd
import numpy as np
# jupyter nbconvert --to script DataPreprocessing_melt.ipynb

___

Melt the dfs - MFCCs

In [3]:
def get_mfccs_per_segment_mean_std(df, meta_cols):
    mfcc_cols = [col for col in df.columns if col.startswith('MFCC_')]
    df_melt = pd.melt(df, id_vars=meta_cols, value_vars=mfcc_cols,
                      var_name='MFCC_feature', value_name='value')
    df_melt[['Session', 'Segment', 'StatType', 'Coefficient']] = df_melt['MFCC_feature']\
        .str.extract(r'MFCC_Session(\d+)_Segment(\d+)_(Mean|Std)Coefficient(\d+)', expand=True)
    df_melt['Session'] = df_melt['Session'].astype(int)
    df_melt['Segment'] = df_melt['Segment'].astype(int)
    df_melt['Coefficient'] = df_melt['Coefficient'].astype(int)

    df_melt['MFCC_label'] = df_melt['StatType'].str.lower() + '_coefficient' + df_melt['Coefficient'].astype(str)

    if len(meta_cols) <= 2:
        index_cols = ['Patient_ID', 'Session', 'Segment', meta_cols[1]]
    else:
        index_cols = ['Patient_ID', 'Session', 'Segment'] + meta_cols[1:]

    # Pivot
    df_long = df_melt.pivot_table(index=index_cols, columns='MFCC_label', values='value', dropna=True).reset_index()
    df_long.columns.name = None

    coef_cols = sorted([col for col in df_long.columns if 'coefficient' in col], key=lambda x: (x.split('_')[0], int(x.split('coefficient')[1])))
    
    new_order = ['Patient_ID', 'Session', 'Segment']
    if len(meta_cols) > 2:
        new_order += meta_cols[1:-1]
    new_order += coef_cols
    if len(meta_cols) >= 2:
        new_order.append(meta_cols[-1])
    df_long = df_long[new_order]
    return df_long

___

Melt the dfs - Embeddings

In [5]:
def get_embeddings_per_segment_mean_std_2D(df, meta_cols):
    embedding_cols = [col for col in df.columns if col.startswith('Embeddings_Session')]
    df_melt = pd.melt(df, id_vars=meta_cols, value_vars=embedding_cols,
                      var_name='Embedding_feature', value_name='value')    
    df_melt[['Session', 'StatType', 'Segment']] = df_melt['Embedding_feature']\
        .str.extract(r'Embeddings_Session(\d+)_(Mean|Std)Segment(\d+)', expand=True)   
    df_melt['Session'] = df_melt['Session'].astype(int)
    df_melt['Segment'] = df_melt['Segment'].astype(int)    
    df_melt['Embedding_label'] = 'embedding_' + df_melt['StatType'].str.lower()    
    if len(meta_cols) <= 2:
        index_cols = ['Patient_ID', 'Session', 'Segment', meta_cols[1]]
    else:
        index_cols = ['Patient_ID', 'Session', 'Segment'] + meta_cols[1:]
    df_long = df_melt.pivot_table(index=index_cols, columns='Embedding_label', values='value', dropna=True).reset_index()
    df_long.columns.name = None
    emb_cols = [col for col in df_long.columns if col.startswith('embedding_')]
    new_order = ['Patient_ID', 'Session', 'Segment']
    if len(meta_cols) > 2:
        new_order += meta_cols[1:-1]
    new_order += emb_cols
    if len(meta_cols) >= 2:
        new_order.append(meta_cols[-1])
    df_long = df_long[new_order]
    return df_long